<a href="https://colab.research.google.com/github/kaifahmad236/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kaifahmad236/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

## Paper finding 1 — Refreshing mature pages

**Finding:** The paper reports that refreshing older/stale pages is associated with higher subsequent impressions. It reports a held-out comparison of refreshed versus stale pages and states that 7 of 9 strata showed statistically significant refresh lift.

**Methodology question:** What exactly defines the outcome and the refresh treatment in the held-out comparison? In particular, I would check whether the post-refresh impression window is completely after the refresh decision and whether pages were assigned to the refreshed and stale groups in a way that avoids selection effects.

**Validation question:** The paper reports held-out comparisons and significance tests, which is useful evidence that the observed difference is not only from the analyzed sample. However, because this is observational data rather than a randomized experiment, the result should be interpreted as an observed association rather than proof that refreshing itself caused the lift.

**My takeaway:** The finding is useful as decision-support for prioritizing refresh candidates, but the evidence does not by itself establish causation.

## Paper finding 2 — Click-through rate falls sharply with lower search position

**Finding:** The paper reports a strong decline in weighted CTR across position tiers: approximately 0.420% for the top 3, 0.340% for positions 4–10, 0.325% for striking distance, 0.163% for positions 21–50, and 0.050% for deeper positions.

**Methodology question:** How is the CTR outcome defined for each position bucket, and are clicks and impressions measured over the same time window for every page? I would also check whether the position buckets contain enough observations and whether weighted CTR is being calculated from total clicks divided by total impressions rather than averaging page-level CTRs.

**Validation question:** The paper reports a large direct comparison across position tiers and also reports a held-out statistical test for Position Tier → CTR. This supports the existence of a measured relationship in the analyzed data. However, position and CTR are naturally related, so the result should be treated as an observed relationship rather than evidence that changing position alone guarantees a particular CTR.

**My takeaway:** Position tier is a useful decision-support signal for prioritizing pages that already have search visibility, but the finding should not be interpreted as a causal guarantee of click improvement.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd

paper_findings = pd.DataFrame([
    {
        "finding": "Refreshing mature pages",
        "reported_evidence": "7 of 9 strata showed statistically significant refresh lift",
        "methodology_question": "Is the post-refresh outcome window fully after the refresh decision, and how were refreshed versus stale pages selected?",
        "claim_level": "Observed association; not causal proof"
    },
    {
        "finding": "CTR falls with lower search position",
        "reported_evidence": "Weighted CTR declines from top-3 to deeper position tiers",
        "methodology_question": "Are clicks and impressions measured over the same window and is weighted CTR calculated consistently?",
        "claim_level": "Measured relationship; not a causal guarantee"
    }
])

display(paper_findings)

,finding,reported_evidence,methodology_question,claim_level
0,Refreshing mature pages,7 of 9 strata showed statistically significant...,Is the post-refresh outcome window fully after...,Observed association; not causal proof
1,CTR falls with lower search position,Weighted CTR declines from top-3 to deeper pos...,Are clicks and impressions measured over the s...,Measured relationship; not a causal guarantee


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

I re-ran the same classification task using the same target definition and the same feature set under two validation designs. The first is a row-level stratified split, which can place pages from the same client in both train and test. The second is a client-grouped split, where all pages from a client stay entirely in either train or test.

The grouped split is the more honest estimate for this task because repeated pages from the same client can share characteristics. Keeping a client entirely on one side reduces the chance that the model is evaluated on information that is too similar to what it saw during training.

I use Precision@50 as the main ranking metric because the practical task is to prioritize a small review queue. I also report average precision and ROC AUC as supporting metrics.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)
from sklearn.model_selection import (
    GroupShuffleSplit,
    train_test_split,
)

RANDOM_STATE = 42

DATA_URL = "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_URL)

numeric_fill_zero = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "content_age_days",
    "age_tier_order",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
    "trend_pct",
]

categorical_features = [
    "competition_level",
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "impression_tier",
    "position_tier",
]

numeric_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "log_impressions_90d",
    "log_clicks_90d",
    "log_sessions_90d",
    "log_ai_sessions_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
]

for column in numeric_fill_zero:
    if column not in df.columns:
        df[column] = 0
    df[column] = pd.to_numeric(df[column], errors="coerce")
    df[column] = df[column].replace([np.inf, -np.inf], np.nan).fillna(0)

for column in categorical_features:
    if column not in df.columns:
        df[column] = "unknown"
    df[column] = df[column].fillna("unknown").astype(str)

df = df[
    (df["impressions_90d"] > 0)
    & (df["content_age_days"] >= 90)
].copy()

df = df.drop_duplicates(subset=["content_id"]).reset_index(drop=True)

df["is_declining_label"] = (
    df["trend_direction"].astype(str).str.lower().eq("down").astype(int)
)

df["log_impressions_90d"] = np.log1p(df["impressions_90d"])
df["log_clicks_90d"] = np.log1p(df["clicks_90d"])
df["log_sessions_90d"] = np.log1p(df["sessions_90d"])
df["log_ai_sessions_90d"] = np.log1p(df["ai_sessions_90d"])

X_numeric = df[numeric_features].copy()

X_categorical = pd.get_dummies(
    df[categorical_features],
    prefix=categorical_features,
    dtype=float
)

X = pd.concat(
    [
        X_numeric.reset_index(drop=True),
        X_categorical.reset_index(drop=True),
    ],
    axis=1,
)

y = df["is_declining_label"].astype(int).reset_index(drop=True)
groups = df["client_id"].fillna("unknown").astype(str).reset_index(drop=True)

model_params = {
    "class_weight": "balanced_subsample",
    "max_depth": 10,
    "min_samples_leaf": 25,
    "n_estimators": 200,
    "n_jobs": -1,
    "random_state": RANDOM_STATE,
}

def precision_at_50(y_true, scores):
    temp = pd.DataFrame({
        "y": np.asarray(y_true),
        "score": np.asarray(scores)
    })
    temp = temp.sort_values("score", ascending=False).head(50)
    return float(temp["y"].mean())

row_train_idx, row_test_idx = train_test_split(
    np.arange(len(df)),
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

row_model = RandomForestClassifier(**model_params)

row_model.fit(
    X.iloc[row_train_idx],
    y.iloc[row_train_idx]
)

row_scores = row_model.predict_proba(
    X.iloc[row_test_idx]
)[:, 1]

row_precision_50 = precision_at_50(
    y.iloc[row_test_idx],
    row_scores
)

row_average_precision = average_precision_score(
    y.iloc[row_test_idx],
    row_scores
)

row_roc_auc = roc_auc_score(
    y.iloc[row_test_idx],
    row_scores
)

group_splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE
)

group_train_idx, group_test_idx = next(
    group_splitter.split(
        X,
        y,
        groups=groups
    )
)

group_model = RandomForestClassifier(**model_params)

group_model.fit(
    X.iloc[group_train_idx],
    y.iloc[group_train_idx]
)

group_scores = group_model.predict_proba(
    X.iloc[group_test_idx]
)[:, 1]

group_precision_50 = precision_at_50(
    y.iloc[group_test_idx],
    group_scores
)

group_average_precision = average_precision_score(
    y.iloc[group_test_idx],
    group_scores
)

group_roc_auc = roc_auc_score(
    y.iloc[group_test_idx],
    group_scores
)

before_after = pd.DataFrame([
    {
        "validation": "Row-stratified split",
        "train_rows": len(row_train_idx),
        "test_rows": len(row_test_idx),
        "test_clients": df.iloc[row_test_idx]["client_id"].nunique(),
        "precision_at_50": row_precision_50,
        "average_precision": row_average_precision,
        "roc_auc": row_roc_auc,
    },
    {
        "validation": "Client-grouped split",
        "train_rows": len(group_train_idx),
        "test_rows": len(group_test_idx),
        "test_clients": df.iloc[group_test_idx]["client_id"].nunique(),
        "precision_at_50": group_precision_50,
        "average_precision": group_average_precision,
        "roc_auc": group_roc_auc,
    }
])

display(before_after)

print(
    "Positive rate:",
    round(float(y.mean()), 4)
)

print(
    "Row-stratified Precision@50:",
    round(row_precision_50, 4)
)

print(
    "Client-grouped Precision@50:",
    round(group_precision_50, 4)
)

,validation,train_rows,test_rows,test_clients,precision_at_50,average_precision,roc_auc
0,Row-stratified split,24000,6000,31,0.90,0.768362,0.757888
1,Client-grouped split,23837,6163,7,0.54,0.589574,0.609623


Positive rate: 0.5421
Row-stratified Precision@50: 0.9
Client-grouped Precision@50: 0.54


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

I checked the final feature set against the three main leakage routes: label-derived information, future or overlapping information, and decision-derived/product flags.

The target is `is_declining_label`, which is derived from `trend_direction`. Therefore `trend_direction` and `trend_pct` are excluded from the model features. Client and content identifiers are also excluded because they are identifiers used for grouping or joining rather than predictive signals.

The model uses measurements such as historical impressions, clicks, sessions, content age, position and engagement variables that are available as observed inputs. Product or decision flags are not included as model features.

The grouped validation also keeps all rows belonging to a client on one side of the evaluation split.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

label_derived_columns = [
    "trend_direction",
    "trend_pct",
    "is_declining_label",
]

identifier_columns = [
    "client_id",
    "content_id",
]

decision_or_product_flags = [
    "health_score",
    "needs_ctr_fix",
    "is_quick_win",
]

used_features = list(X.columns)

leakage_results = []

for column in label_derived_columns:
    leakage_results.append({
        "check": "Label-derived feature",
        "field": column,
        "present_in_model": column in used_features,
        "verdict": "PASS" if column not in used_features else "FAIL"
    })

for column in identifier_columns:
    leakage_results.append({
        "check": "Identifier used as feature",
        "field": column,
        "present_in_model": column in used_features,
        "verdict": "PASS" if column not in used_features else "FAIL"
    })

for column in decision_or_product_flags:
    leakage_results.append({
        "check": "Decision/product flag",
        "field": column,
        "present_in_model": column in used_features,
        "verdict": "PASS" if column not in used_features else "FAIL"
    })

leakage_table = pd.DataFrame(leakage_results)

display(leakage_table)

print("Number of model features:", len(used_features))
print(
    "Label-derived fields used as features:",
    [
        c for c in label_derived_columns
        if c in used_features
    ]
)

print(
    "Identifier fields used as features:",
    [
        c for c in identifier_columns
        if c in used_features
    ]
)

print(
    "Decision/product flags used as features:",
    [
        c for c in decision_or_product_flags
        if c in used_features
    ]
)

assert not any(
    c in used_features
    for c in label_derived_columns
)

assert not any(
    c in used_features
    for c in identifier_columns
)

assert not any(
    c in used_features
    for c in decision_or_product_flags
)

print("Leakage audit passed for the checked feature categories.")

,check,field,present_in_model,verdict
0,Label-derived feature,trend_direction,False,PASS
1,Label-derived feature,trend_pct,False,PASS
2,Label-derived feature,is_declining_label,False,PASS
3,Identifier used as feature,client_id,False,PASS
4,Identifier used as feature,content_id,False,PASS
5,Decision/product flag,health_score,False,PASS
6,Decision/product flag,needs_ctr_fix,False,PASS
7,Decision/product flag,is_quick_win,False,PASS


Number of model features: 52
Label-derived fields used as features: []
Identifier fields used as features: []
Decision/product flags used as features: []
Leakage audit passed for the checked feature categories.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### Original claim

The model can identify declining pages and reliably tell us which pages should be refreshed.

### Evidence-based rewrite

The model produced a measured ranking signal for pages associated with the declining label. Performance was higher/lower under the client-grouped split than under the row-level split, showing that the validation design affects the measured result.

The grouped result is the more appropriate estimate for decision-support because pages from the same client are kept on the same side of the evaluation split. The model should therefore be treated as a directional prioritization tool rather than evidence that a particular page will decline or that refreshing a page will cause recovery.

The observed results do not establish causation. They support using the model as decision-support for prioritizing pages for human review, subject to the measured validation performance and the leakage checks above.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

precision_change = group_precision_50 - row_precision_50
average_precision_change = group_average_precision - row_average_precision
roc_auc_change = group_roc_auc - row_roc_auc

claim_summary = pd.DataFrame([
    {
        "metric": "Precision@50",
        "row_split": row_precision_50,
        "grouped_split": group_precision_50,
        "grouped_minus_row": precision_change
    },
    {
        "metric": "Average precision",
        "row_split": row_average_precision,
        "grouped_split": group_average_precision,
        "grouped_minus_row": average_precision_change
    },
    {
        "metric": "ROC AUC",
        "row_split": row_roc_auc,
        "grouped_split": group_roc_auc,
        "grouped_minus_row": roc_auc_change
    }
])

display(claim_summary)

if precision_change < 0:
    validation_statement = (
        "Precision@50 decreased under client-grouped validation, "
        "so the row-level result appears more optimistic than the "
        "more conservative grouped estimate."
    )
elif precision_change > 0:
    validation_statement = (
        "Precision@50 increased under client-grouped validation, "
        "but the grouped result should still be treated as the "
        "more appropriate estimate for this client-grouped question."
    )
else:
    validation_statement = (
        "Precision@50 was unchanged between the two validation designs."
    )

print(validation_statement)

print(
    "The grouped split keeps each client's rows entirely in either "
    "training or test data."
)

print(
    "The results support directional decision-support, not causal claims."
)

,metric,row_split,grouped_split,grouped_minus_row
0,Precision@50,0.900000,0.540000,-0.360000
1,Average precision,0.768362,0.589574,-0.178789
2,ROC AUC,0.757888,0.609623,-0.148265


Precision@50 decreased under client-grouped validation, so the row-level result appears more optimistic than the more conservative grouped estimate.
The grouped split keeps each client's rows entirely in either training or test data.
The results support directional decision-support, not causal claims.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.